# 04 - LLM Reranker RAG

**Requires GROQ_API_KEY.**

BM25 is good at finding chunks that share words with the query, but it has
no understanding of meaning. An LLM reranker reads each retrieved chunk and
scores it by relevance before the final answer is generated.

### The pipeline

```
Query
  |
  v
BM25 retrieves top-10 candidates  (fast keyword search)
  |
  v
LLM scores each candidate 0-10    (slow but semantic)
  |
  v
Keep top-3 by LLM score
  |
  v
Groq generates grounded answer
```

### What you will learn

| Step | Concept |
|------|---------|
| 1 | Retrieve many candidates with BM25 |
| 2 | Ask the LLM to score each candidate for relevance |
| 3 | Select the top-k by LLM score |
| 4 | Generate an answer using only the reranked chunks |
| 5 | Compare answers with and without reranking |

> Requires: GROQ_API_KEY in a .env file

## 1. Install and Import

In [ ]:
# !pip install rank-bm25 langchain-groq python-dotenv

In [ ]:
import re, os, json
from pathlib import Path
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 2. Build the Chunk Index

In [ ]:
DATA_DIR = Path("../data")

STOP_WORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "of",
    "and", "or", "but", "for", "with", "by", "from", "as", "was",
    "are", "were", "be", "been", "has", "have", "had", "this", "that",
    "these", "those", "its", "their", "they", "also", "which", "into",
    "more", "than", "about", "such", "can", "not", "over", "up", "out"
}

def tokenize(text):
    text = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    return [t for t in text.split() if t not in STOP_WORDS and len(t) > 2]

def sliding_window_chunks(text, source, window=150, stride=75):
    words, chunks, start = text.split(), [], 0
    while start < len(words):
        end   = min(start + window, len(words))
        chunk = " ".join(words[start:end])
        if len(chunk) >= 80:
            chunks.append({"source": source, "text": chunk})
        if end == len(words):
            break
        start += stride
    return chunks


all_chunks = []
for fp in sorted(DATA_DIR.glob("*.txt")):
    all_chunks.extend(sliding_window_chunks(fp.read_text(), fp.name))

tokenized = [tokenize(c["text"]) for c in all_chunks]
bm25      = BM25Okapi(tokenized)
llm       = ChatGroq(model="llama3-8b-8192", temperature=0.0)

print(f"Indexed {len(all_chunks)} chunks")

## 3. Stage 1 - BM25 Candidate Retrieval

In [ ]:
def bm25_candidates(query, top_n=10):
    """Fast keyword retrieval: return top_n candidates."""    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(scores, all_chunks), key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in ranked[:top_n]]


# Demonstrate on one query
query = "What causes Arctic ice to melt faster?"
candidates = bm25_candidates(query, top_n=10)

print(f"BM25 returned {len(candidates)} candidates:")
for i, c in enumerate(candidates, 1):
    print(f"  [{i:2d}] {c['source']:35s}  {c['text'][:70]}...")

## 4. Stage 2 - LLM Reranker

The LLM reads each candidate and assigns a relevance score from 0 to 10.
We ask it to respond with only a JSON object so the score is easy to parse.

In [ ]:
RERANK_SYSTEM = (
    "You are a relevance judge. "
    "Given a query and a text passage, respond with a JSON object "
    "containing a single key 'score' with an integer value from 0 to 10. "
    "10 means the passage directly answers the query. "
    "0 means the passage is completely unrelated. "
    "Respond with only the JSON object, nothing else."
)

def score_chunk(query, chunk_text):
    """Ask the LLM to score one chunk. Return an integer 0-10."""    prompt = f"Query: {query}\n\nPassage:\n{chunk_text[:500]}"
    response = llm.invoke([
        SystemMessage(content=RERANK_SYSTEM),
        HumanMessage(content=prompt),
    ])
    raw = response.content.strip()
    try:
        # Extract JSON from response
        start = raw.index("{")
        end   = raw.rindex("}") + 1
        data  = json.loads(raw[start:end])
        return int(data.get("score", 0))
    except Exception:
        return 0   # fallback: score 0 if parsing fails


def rerank(query, candidates):
    """Score all candidates and return them sorted by LLM score."""    scored = []
    for chunk in candidates:
        score = score_chunk(query, chunk["text"])
        scored.append((score, chunk))
        print(f"  Scored {score:2d}/10 - {chunk['source']:30s}  {chunk['text'][:60]}...")
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored

print("Reranker ready!")

## 5. Full Pipeline: Retrieve, Rerank, Generate

In [ ]:
GENERATE_SYSTEM = (
    "You are a factual assistant. Answer the question using only "
    "the provided context. If the answer is not in the context, say so. "
    "Keep your answer concise."
)

def answer_with_reranking(query, bm25_top_n=10, final_top_k=3):
    """Full pipeline: BM25 -> rerank -> generate."""    print(f"Query: {query}")
    print("=" * 60)

    # Stage 1: BM25 candidate retrieval
    print(f"\nStage 1: BM25 retrieves top-{bm25_top_n} candidates...")
    candidates = bm25_candidates(query, top_n=bm25_top_n)

    # Stage 2: LLM reranking
    print(f"\nStage 2: LLM scores each candidate...")
    ranked = rerank(query, candidates)

    # Keep top-k by LLM score
    top_chunks = [chunk for _, chunk in ranked[:final_top_k]]

    print(f"\nTop-{final_top_k} after reranking:")
    for score, chunk in ranked[:final_top_k]:
        print(f"  Score {score}/10  {chunk['source']}")

    # Stage 3: Answer generation
    context = "\n\n".join(
        f"[{c['source']}]\n{c['text']}" for c in top_chunks
    )
    response = llm.invoke([
        SystemMessage(content=GENERATE_SYSTEM),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ])

    print(f"\nAnswer:\n{response.content.strip()}")
    return response.content.strip()

answer_with_reranking("What caused the decline of the Indus Valley Civilization?")

In [ ]:
answer_with_reranking("How does ocean acidification affect marine life?")

## 6. Compare: With and Without Reranking

This cell runs both approaches on the same query so you can see the difference
in which chunks are selected.

In [ ]:
COMPARE_QUERY = "What is overfitting in machine learning and how is it prevented?"

print("WITHOUT reranking (BM25 top-3 only):")
print("-" * 55)
bm25_top3 = bm25_candidates(COMPARE_QUERY, top_n=3)
for i, c in enumerate(bm25_top3, 1):
    print(f"  [{i}] {c['source']:35s}  {c['text'][:80]}...")

print()
print("WITH reranking (BM25 top-10, LLM selects top-3):")
print("-" * 55)
candidates = bm25_candidates(COMPARE_QUERY, top_n=10)
ranked     = rerank(COMPARE_QUERY, candidates)
for score, chunk in ranked[:3]:
    print(f"  [{score}/10] {chunk['source']:30s}  {chunk['text'][:80]}...")

## 7. Key Takeaways

| Stage | Method | Speed | Quality |
|-------|--------|-------|---------|
| Retrieval | BM25 | Fast (milliseconds) | Good keyword match |
| Reranking | LLM scoring | Slow (one LLM call per chunk) | Understands meaning |
| Generation | Groq LLM | Fast | Grounded in selected context |

**When to use a reranker:**
- Your queries use synonyms or paraphrases not in the documents
- Keyword overlap gives the wrong results
- You can afford the extra LLM calls (about 1 call per candidate)

**Next notebook:** Map-Reduce RAG, where every chunk is summarized by the LLM
and then the summaries are combined into a final answer.